<a href="https://colab.research.google.com/github/Marcin19721205/Timeseries_Data_Processing_Basic/blob/main/!!!Simulink_Dynamic_LSTM_SARIMAX_RFR_Prophet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Keras / TF (modelowanie)
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Time series utils (okna czasowe / dataset z sekwencji)
from tensorflow.keras.utils import timeseries_dataset_from_array

# Plotly (wykresy)
import plotly.express as px
import plotly.graph_objects as go

import pandas as pd  # data table #


Bez standaryzacji - wyniki są blisko siebie

In [2]:
# Foreword: re-read CSV with the correct separator and numeric format, then coerce to numeric and preview. #

import pandas as pd  # data table #

path = "sample_data/Simulink_Timeseries_01.csv"  # file location #

df = pd.read_csv(path, sep=";", decimal=".", encoding="utf-8-sig")  # split columns by ';' and parse floats with '.' #

df.columns = df.columns.str.strip()  # clean header whitespace #
df = df.apply(pd.to_numeric, errors="coerce")  # force numeric (non-numeric -> NaN) #

print(df.shape)  # (rows, cols) #
display(df.head(10))  # preview #
display(df.isna().sum())  # quick sanity: NaNs per column #



(6001, 8)


,TIME,X1OUT,X1SP,X2OUT,X2SP,X3OUT,X3SP,Y
0,0,-0.002233,0.3,-0.000541,0.8,0.001589,0.3,0.000012
1,1,0.012668,0.3,0.006050,0.8,0.019859,0.3,0.000726
2,2,0.026283,0.3,0.014362,0.8,0.056131,0.3,0.002747
3,3,0.037083,0.3,0.036937,0.8,0.089217,0.3,0.006508
4,4,0.053628,0.3,0.056952,0.8,0.120559,0.3,0.012090
5,5,0.066516,0.3,0.085733,0.8,0.146780,0.3,0.019350
6,6,0.080481,0.3,0.115617,0.8,0.166958,0.3,0.028114
7,7,0.087747,0.3,0.154050,0.8,0.194496,0.3,0.038104
8,8,0.102472,0.3,0.191832,0.8,0.219550,0.3,0.049482
9,9,0.110620,0.3,0.233669,0.8,0.245548,0.3,0.062242


,0
TIME,0
X1OUT,0
X1SP,0
X2OUT,0
X2SP,0
X3OUT,0
X3SP,0
Y,0


In [3]:
# Foreword: set TIME as the time index (sorted), making the DataFrame ready for time-series ops. #

df = df.set_index("TIME")  # TIME -> index #
df.index = pd.to_numeric(df.index, errors="coerce")  # ensure numeric time axis #
df = df[~df.index.isna()].sort_index()  # drop bad TIME rows + sort by time #
df.index.name = "TIME"  # keep index name #

display(df.head(10))  # preview #
print(df.index.min(), df.index.max(), df.index.is_monotonic_increasing)  # range + monotonic check #


,X1OUT,X1SP,X2OUT,X2SP,X3OUT,X3SP,Y
TIME,,,,,,,
0,-0.002233,0.3,-0.000541,0.8,0.001589,0.3,0.000012
1,0.012668,0.3,0.006050,0.8,0.019859,0.3,0.000726
2,0.026283,0.3,0.014362,0.8,0.056131,0.3,0.002747
3,0.037083,0.3,0.036937,0.8,0.089217,0.3,0.006508
4,0.053628,0.3,0.056952,0.8,0.120559,0.3,0.012090
5,0.066516,0.3,0.085733,0.8,0.146780,0.3,0.019350
6,0.080481,0.3,0.115617,0.8,0.166958,0.3,0.028114
7,0.087747,0.3,0.154050,0.8,0.194496,0.3,0.038104
8,0.102472,0.3,0.191832,0.8,0.219550,0.3,0.049482


0 6000 True


In [4]:
# Foreword: plot X1OUT and X1SP vs TIME using Plotly (dark), from the TIME index. #

import plotly.graph_objects as go  # plotly low-level #

fig = go.Figure()  # init #
fig.add_scatter(x=df.index, y=df["X1OUT"], mode="lines", name="X1OUT")  # output #
fig.add_scatter(x=df.index, y=df["X1SP"],  mode="lines", name="X1SP")   # setpoint #
fig.update_layout(template="plotly_dark", xaxis_title="TIME", yaxis_title="Value")  # labels + dark #
fig.show()  # render #


In [5]:
# Foreword: plot X2OUT and X2SP vs TIME using Plotly (dark), from the TIME index. #

import plotly.graph_objects as go  # plotly low-level #

fig = go.Figure()  # init #
fig.add_scatter(x=df.index, y=df["X2OUT"], mode="lines", name="X2OUT")  # output #
fig.add_scatter(x=df.index, y=df["X2SP"],  mode="lines", name="X2SP")   # setpoint #
fig.update_layout(template="plotly_dark", xaxis_title="TIME", yaxis_title="Value")  # labels + dark #
fig.show()  # render #


In [6]:
# Foreword: plot X3OUT and X3SP vs TIME using Plotly (dark), from the TIME index. #

import plotly.graph_objects as go  # plotly low-level #

fig = go.Figure()  # init #
fig.add_scatter(x=df.index, y=df["X3OUT"], mode="lines", name="X3OUT")  # output #
fig.add_scatter(x=df.index, y=df["X3SP"],  mode="lines", name="X3SP")   # setpoint #
fig.update_layout(template="plotly_dark", xaxis_title="TIME", yaxis_title="Value")  # labels + dark #
fig.show()  # render #


In [7]:
# Foreword: plot Y vs TIME using Plotly (dark), from the TIME index. #

import plotly.graph_objects as go  # plotly low-level #

fig = go.Figure()  # init #
fig.add_scatter(x=df.index, y=df["Y"], mode="lines", name="Y")  # Y over time #
fig.update_layout(template="plotly_dark", xaxis_title="TIME", yaxis_title="Y")  # labels + dark #
fig.show()  # render #


Split Train / Test

In [8]:
# Foreword: time-based split (no shuffle) -> train = first half, test = second half, using TIME index. #

N = len(df)  # number of samples #
cut = N // 2  # 50/50 split point (time-ordered) #

train_df = df.iloc[:cut].copy()  # older data -> train #
test_df  = df.iloc[cut:].copy()  # newer data -> test #

X_train = train_df[["X1OUT", "X1SP"]].copy()  # features (example) #
y_train = train_df["Y"].copy()  # target #

X_test  = test_df[["X1OUT", "X1SP"]].copy()  # features (example) #
y_test  = test_df["Y"].copy()  # target #

print(train_df.shape, test_df.shape)  # sanity shapes #
print(train_df.index.min(), train_df.index.max())  # train time range #
print(test_df.index.min(),  test_df.index.max())   # test time range #


(3000, 7) (3001, 7)
0 2999
3000 6000


Dataset - 100 samples

In [9]:
# Foreword: build LSTM dataset (sliding windows) with T=100 timesteps, D=6 features, label = next-step Y. #

import numpy as np  # arrays #

T = 100  # window length (timesteps) #
D = 6    # number of features per timestep #

feat_cols = ["X1OUT","X1SP","X2OUT","X2SP","X3OUT","X3SP"]  # features #
target_col = "Y"  # label #

X_all = df[feat_cols].to_numpy(dtype=np.float32)  # (N, 6) #
y_all = df[target_col].to_numpy(dtype=np.float32)  # (N,) #

X = np.lib.stride_tricks.sliding_window_view(X_all, (T, D))[:, 0, :, :]  # (N-T+1, T, 6) #
Y = y_all[T:]  # next-step label aligned to window end -> (N-T,) #
X = X[:-1]  # align to Y length -> (N-T, T, 6) #

N = len(X)  # number of samples #
print("X.shape", X.shape, "Y.shape", Y.shape)  # sanity #


X.shape (5901, 100, 6) Y.shape (5901,)


#LSTM Model

In [10]:
# Foreword: LSTM regressor for next-step Y from window (T, D=6); bigger than the example, linear output. #

from tensorflow.keras import Model  # functional API #
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout  # layers #
from tensorflow.keras.optimizers import Adam  # optimizer #

i = Input(shape=(T, D))  # input: (timesteps, features) = (T,6) #
x = LSTM(64, return_sequences=True)(i)  # sequence -> sequence (richer memory) #
x = Dropout(0.2)(x)  # regularize #
x = LSTM(32)(x)  # sequence -> vector #
x = Dense(32, activation="relu")(x)  # nonlinear head #
x = Dense(1)(x)  # linear output for regression #
# Foreword: same network, just named model1. #

model1 = Model(i, x)  # build model #
model1.compile(loss="mse", optimizer=Adam(learning_rate=0.001), metrics=["mae","mape"])  # compile #
model1.summary()  # inspect #



Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 100, 6)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 100, 64)        │        18,176 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 100, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         1,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 31,681 (123.75 KB)

 Trainable params: 31,681 (123.75 KB)

 Non-trainable params: 0 (0.00 B)

#Train LSTM

In [11]:
# Foreword: train model1 time-wise (no shuffle), reasonable hyperparams for LSTM regression, keep history in history1. #

from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau  # callbacks #

N = len(X)  # samples #
split = N // 2  # time split 50/50 (train/val) #

cb = [
    EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True),  # stop before overfit #
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=7, min_lr=1e-5)  # auto LR tuning #
]

history1 = model1.fit(
    X[:split], Y[:split],  # train (older) #
    validation_data=(X[split:], Y[split:]),  # val (newer) #
    epochs=200,  # upper cap; ES will stop earlier #
    batch_size=32,  # stable default for LSTM #
    shuffle=False,  # keep temporal order #
    callbacks=cb,  # training control #
    verbose=1  # logs #
)


Epoch 1/200
93/93 ━━━━━━━━━━━━━━━━━━━━ 7s 25ms/step - loss: 0.0556 - mae: 0.1800 - mape: 41.7810 - val_loss: 0.0038 - val_mae: 0.0536 - val_mape: 19.8447 - learning_rate: 0.0010
Epoch 2/200
93/93 ━━━━━━━━━━━━━━━━━━━━ 2s 20ms/step - loss: 0.0030 - mae: 0.0430 - mape: 10.2887 - val_loss: 0.0230 - val_mae: 0.1486 - val_mape: 45.3167 - learning_rate: 0.0010
Epoch 3/200
93/93 ━━━━━━━━━━━━━━━━━━━━ 3s 28ms/step - loss: 0.0069 - mae: 0.0689 - mape: 16.1982 - val_loss: 0.0020 - val_mae: 0.0355 - val_mape: 13.9672 - learning_rate: 0.0010
Epoch 4/200
93/93 ━━━━━━━━━━━━━━━━━━━━ 4s 20ms/step - loss: 0.0017 - mae: 0.0324 - mape: 7.8196 - val_loss: 7.7717e-04 - val_mae: 0.0181 - val_mape: 5.4248 - learning_rate: 0.0010
Epoch 5/200
93/93 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - loss: 0.0010 - mae: 0.0257 - mape: 5.9900 - val_loss: 0.0011 - val_mae: 0.0244 - val_mape: 5.9148 - learning_rate: 0.0010
Epoch 6/200
93/93 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - loss: 6.4821e-04 - mae: 0.0200 - mape: 4.6695 - val_loss

#LSTM TRain Metrics

In [12]:
# Foreword: Plotly dark plots for training/validation loss and metrics from history1 (handles mae/mape as "accuracy"). #

import pandas as pd  # history to df #
import plotly.graph_objects as go  # plotly #

h = pd.DataFrame(history1.history)  # metrics table #

# loss #
fig = go.Figure()  # init #
fig.add_scatter(y=h["loss"], mode="lines", name="loss")  # train loss #
fig.add_scatter(y=h["val_loss"], mode="lines", name="val_loss")  # val loss #
fig.update_layout(template="plotly_dark", title="Loss", xaxis_title="Epoch", yaxis_title="MSE")  # layout #
fig.show()  # render #

# "accuracy" substitute: mae (and/or mape) #
metric = "mae" if "mae" in h.columns else ("mape" if "mape" in h.columns else None)  # pick available #
if metric:  # only if present #
    fig = go.Figure()  # init #
    fig.add_scatter(y=h[metric], mode="lines", name=metric)  # train metric #
    fig.add_scatter(y=h[f"val_{metric}"], mode="lines", name=f"val_{metric}")  # val metric #
    fig.update_layout(template="plotly_dark", title=f"{metric.upper()} (as accuracy proxy)", xaxis_title="Epoch", yaxis_title=metric.upper())  # layout #
    fig.show()  # render #


one-step na teście (bez recursive)

In [14]:
# Foreword: proper test evaluation -> predict Y from real X windows (no recursive feedback). #

import numpy as np  # arrays #
import plotly.graph_objects as go  # plots #

N = len(X)  # samples #
split = N // 2  # time split #

test_target = Y[split:]  # true #
test_pred = model1.predict(X[split:], verbose=1).reshape(-1)  # predicted #

fig = go.Figure()  # init #
fig.add_scatter(y=test_target, mode="lines", name="Y_true")  # true #
fig.add_scatter(y=test_pred,   mode="lines", name="Y_pred")  # pred #
fig.update_layout(template="plotly_dark", title="Test: one-step forecast", xaxis_title="Test step", yaxis_title="Y")  # layout #
fig.show()  # render #


93/93 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step


rolling, prawdziwymi cechami z testu

In [16]:
# Foreword: rolling over test horizon using real future X windows (no fake feature injection). #

import numpy as np  # arrays #
import plotly.graph_objects as go  # plots #

N = len(X)  # samples #
split = N // 2  # time split #

test_target = Y[split:]  # true #
preds = []  # predictions #

for k in range(len(test_target)):  # walk forward #
    xk = X[split + k]  # real test window (T,6) #
    p = float(model1.predict(xk.reshape(1, T, D), verbose=1)[0, 0])  # 1-step pred #
    preds.append(p)  # collect #

preds = np.array(preds, dtype=np.float32)  # to array #

fig = go.Figure()  # init #
fig.add_scatter(y=test_target, mode="lines", name="Y_true")  # true #
fig.add_scatter(y=preds,       mode="lines", name="Y_pred")  # pred #
fig.update_layout(template="plotly_dark", title="Test: walk-forward (real X)", xaxis_title="Test step", yaxis_title="Y")  # layout #
fig.show()  # render #


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━

# pobranie ustawień wynik predykcji ze zbioru dataset do sprawdzenia predykcji

In [27]:
# Foreword: pick sample n, extract feature values at time n, build baseline window from them, forecast Y at horizon H=100. #

import numpy as np  # arrays #

n = 245  # sample index in df (time-ordered); set your n #

# extract baseline feature values at time n #
row = df.iloc[n]  # one sample #
X1OUT_0 = float(row["X1OUT"])  # value at n #
X1SP_0  = float(row["X1SP"])   # value at n #
X2OUT_0 = float(row["X2OUT"])  # value at n #
X2SP_0  = float(row["X2SP"])   # value at n #
X3OUT_0 = float(row["X3OUT"])  # value at n #
X3SP_0  = float(row["X3SP"])   # value at n #

print("n =", n)  # info #
print("X1OUT_0 =", X1OUT_0, "X1SP_0 =", X1SP_0)  # display #
print("X2OUT_0 =", X2OUT_0, "X2SP_0 =", X2SP_0)  # display #
print("X3OUT_0 =", X3OUT_0, "X3SP_0 =", X3SP_0)  # display #

# build constant window from baseline (training feature order!) #
u0 = np.array([X1OUT_0, X1SP_0, X2OUT_0, X2SP_0, X3OUT_0, X3SP_0], dtype=np.float32)  # (6,) #
last_x = np.tile(u0, (T, 1))  # (T,6) #

H = 100  # horizon steps #
preds = np.empty(H, dtype=np.float32)  # forecast path #

for t in range(H):  # horizon walk #
    preds[t] = float(model1.predict(last_x.reshape(1, T, D), verbose=0)[0, 0])  # Y_hat #
    last_x = np.roll(last_x, -1, axis=0)  # shift #
    last_x[-1, :] = u0  # keep inputs constant at baseline #

print("Y_pred(+100) =", float(preds[-1]))  # prediction 100 steps ahead #


n = 245
X1OUT_0 = 0.926924298471125 X1SP_0 = 1.0
X2OUT_0 = 0.127729274186114 X2SP_0 = 0.2
X3OUT_0 = 0.196273003792552 X3SP_0 = 0.2
Y_pred(+100) = 0.42435652017593384


#Predykcja LSTM na oknie

In [44]:
# Foreword: build step inputs (H=100), forecast Y horizon with model1, and plot Y dynamics (Plotly dark). #

import numpy as np  # arrays #
import plotly.graph_objects as go  # plots #

H = 100  # horizon length #
t0 = 10  # step time (0..H-1); set 0 for immediate step #

# step levels (before -> after) #
X1OUT_0, X1OUT_1 = 0.93, 0.93  # OUT1 #
X2OUT_0, X2OUT_1 = 0.12, 0.12  # OUT2 #
X3OUT_0, X3OUT_1 = 0.20, 0.20  # OUT3 #
X1SP_0,  X1SP_1  = 1.0, 0.8  # SP1 #
X2SP_0,  X2SP_1  = 0.2, 0.4  # SP2 #
X3SP_0,  X3SP_1  = 0.2, 0.4  # SP3 #

# build horizon arrays #
X1OUT_f = np.r_[np.full(t0, X1OUT_0), np.full(H-t0, X1OUT_1)].astype(np.float32)  # (H,) #
X2OUT_f = np.r_[np.full(t0, X2OUT_0), np.full(H-t0, X2OUT_1)].astype(np.float32)  # (H,) #
X3OUT_f = np.r_[np.full(t0, X3OUT_0), np.full(H-t0, X3OUT_1)].astype(np.float32)  # (H,) #
X1SP_f  = np.r_[np.full(t0, X1SP_0 ), np.full(H-t0, X1SP_1 )].astype(np.float32)  # (H,) #
X2SP_f  = np.r_[np.full(t0, X2SP_0 ), np.full(H-t0, X2SP_1 )].astype(np.float32)  # (H,) #
X3SP_f  = np.r_[np.full(t0, X3SP_0 ), np.full(H-t0, X3SP_1 )].astype(np.float32)  # (H,) #

# user order -> (H,6) #
F_user = np.column_stack([X1OUT_f, X2OUT_f, X3OUT_f, X1SP_f, X2SP_f, X3SP_f]).astype(np.float32)  # (H,6) #
F = F_user[:, [0, 3, 1, 4, 2, 5]]  # reorder to training: [X1OUT,X1SP,X2OUT,X2SP,X3OUT,X3SP] #

#
#
# wpisuje wartości Xn_OUT, XnSP to startowych od których rozpoczyna dynamikę
u0 = np.array([X1OUT_0, X1SP_0, X2OUT_0, X2SP_0, X3OUT_0, X3SP_0], dtype=np.float32)  # baseline row #
last_x = np.tile(u0, (T, 1))  # (T,6) #
#
#
#

preds = np.empty(H, dtype=np.float32)  # horizon preds #

for t in range(H):  # walk horizon #
    preds[t] = float(model1.predict(last_x.reshape(1, T, D), verbose=0)[0, 0])  # Y_hat #
    last_x = np.roll(last_x, -1, axis=0)  # shift window #
    last_x[-1, :] = F[t, :]  # append provided features #

# plot Y dynamics #
k = np.arange(H)  # x axis #
fig = go.Figure()  # init #
fig.add_scatter(x=k, y=preds, mode="lines", name="Y_pred")  # Y forecast #
fig.add_vline(x=t0, line_dash="dash")  # step moment marker #
fig.update_layout(template="plotly_dark", title="Y dynamics (forecast horizon H=100) | unit-step OUT/SP on step scenario", xaxis_title="k (step ahead)", yaxis_title="Y")  # layout #
fig.show()  # render #



#Arimax / SARIMAX

przygotowanie danych pod ARIMAX/SARIMAX - Y jako szereg + 6 sygnałów jako exog i czasowy split 50/50

In [30]:
# Foreword: SARIMAX/ARIMAX setup (model2) -> build Y and exog, time-split 50/50, align indices. #

import numpy as np  # arrays #
import pandas as pd  # tables #
from statsmodels.tsa.statespace.sarimax import SARIMAX  # ARIMAX/SARIMAX #

y2 = df["Y"].astype(float)  # target series #
X2 = df[["X1OUT","X1SP","X2OUT","X2SP","X3OUT","X3SP"]].astype(float)  # exog features #

mask2 = y2.notna() & X2.notna().all(axis=1)  # drop NaNs consistently #
y2 = y2[mask2]  # clean y #
X2 = X2[mask2]  # clean X #

N2 = len(y2)  # samples #
split2 = N2 // 2  # time split #

y_train2, y_test2 = y2.iloc[:split2], y2.iloc[split2:]  # train/test y #
X_train2, X_test2 = X2.iloc[:split2], X2.iloc[split2:]  # train/test exog #

print(y_train2.shape, y_test2.shape, X_train2.shape, X_test2.shape)  # sanity #


(3000,) (3001,) (3000, 6) (3001, 6)


Test Dickeya - Fullera H1 - stacjonarny

In [31]:
# Foreword: step 2 -> check stationarity of y_train2 with ADF, pick differencing d2 (0/1) as a baseline for model2. #

from statsmodels.tsa.stattools import adfuller  # ADF test #

adf_stat2, pval2, _, _, crit2, _ = adfuller(y_train2.dropna(), autolag="AIC")  # ADF on train only #

d2 = 0 if pval2 < 0.05 else 1  # if non-stationary -> difference once #

print("ADF stat:", adf_stat2)  # test statistic #
print("p-value:", pval2)  # stationarity decision #
print("crit:", crit2)  # critical values #
print("chosen d2:", d2)  # differencing order #


ADF stat: -3.530642537533021
p-value: 0.007234132477671903
crit: {'1%': np.float64(-3.432537472983712), '5%': np.float64(-2.8625064838167327), '10%': np.float64(-2.5672844849053806)}
chosen d2: 0


In [32]:
# Foreword: step 3 -> fit a small SARIMAX (ARIMAX) grid on TRAIN to choose (p,q) with d2=0, then refit best as model2. #

import warnings  # silence warnings #
from statsmodels.tsa.statespace.sarimax import SARIMAX  # model #

warnings.filterwarnings("ignore")  # cleaner output #

best_aic2 = np.inf  # init #
best_order2 = None  # init #

for p2 in range(0, 4):  # p=0..3 #
    for q2 in range(0, 4):  # q=0..3 #
        try:
            m = SARIMAX(
                y_train2, exog=X_train2,  # train only #
                order=(p2, d2, q2),  # ARIMA(p,d,q) #
                enforce_stationarity=False, enforce_invertibility=False  # robust fit #
            ).fit(disp=False)  # fit #
            if m.aic < best_aic2:  # keep best #
                best_aic2, best_order2 = m.aic, (p2, d2, q2)  # store #
        except Exception:
            pass  # skip failing combos #

print("best_order2:", best_order2, "best_aic2:", best_aic2)  # report #

model2 = SARIMAX(
    y_train2, exog=X_train2,  # train only #
    order=best_order2,  # best order #
    enforce_stationarity=False, enforce_invertibility=False  # robust fit #
).fit(disp=False)  # final fit #

print(model2.summary())  # inspect #


best_order2: (2, 0, 3) best_aic2: -36911.4089512807
                               SARIMAX Results                                
Dep. Variable:                      Y   No. Observations:                 3000
Model:               SARIMAX(2, 0, 3)   Log Likelihood               18467.704
Date:                Sun, 18 Jan 2026   AIC                         -36911.409
Time:                        12:13:03   BIC                         -36839.349
Sample:                             0   HQIC                        -36885.487
                               - 3000                                         
Covariance Type:                  opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
X1OUT         -0.0185      0.001    -12.858      0.000      -0.021      -0.016
X1SP          -0.0025      0.000    -22.680      0.000      -0.003      -0.002


SARIMAX bez lagów - bardzo słabo. muszą być lagi

In [34]:
# Foreword: predict ONLY first H=100 points of the TEST set (not the whole 3000), compute metrics, Plotly dark. #

import numpy as np  # arrays #
import plotly.graph_objects as go  # plots #

H = 100  # number of test points to forecast #

y_test2_h = y_test2.iloc[:H]  # first 100 true points #
X_test2_h = X_test2.iloc[:H]  # first 100 exog rows #

pred2_h = model2.get_forecast(steps=H, exog=X_test2_h)  # 100-step forecast from test start #
yhat2_h = pred2_h.predicted_mean  # preds aligned 0..H-1 #
history2 = pred2_h  # store #

err2_h = (y_test2_h.values - yhat2_h.values)  # residuals #
mae2_h = float(np.mean(np.abs(err2_h)))  # MAE #
mape2_h = float(np.nanmean(np.abs(err2_h / np.where(y_test2_h.values==0, np.nan, y_test2_h.values))) * 100)  # MAPE (%) #
mse2_h = float(np.mean(err2_h**2))  # MSE #

print("H:", H, "MAE2:", mae2_h, "MAPE2[%]:", mape2_h, "MSE2:", mse2_h)  # metrics #

fig = go.Figure()  # init #
fig.add_scatter(x=y_test2_h.index, y=y_test2_h.values, mode="lines", name="Y_true")  # true #
fig.add_scatter(x=y_test2_h.index, y=yhat2_h.values,   mode="lines", name="Y_pred")  # pred #
fig.update_layout(template="plotly_dark", title=f"SARIMAX test forecast H={H} | order={best_order2} | MAE={mae2_h:.4f} MAPE={mape2_h:.2f}% MSE={mse2_h:.6f}", xaxis_title="TIME", yaxis_title="Y")  # layout #
fig.show()  # render #


H: 100 MAE2: 0.15257131121462708 MAPE2[%]: 25.37028405446383 MSE2: 0.02632172624473551


Sarimax z lagami - bardzo kosztowne obliczeniowo

In [35]:
# Foreword: step 5 -> add exog lags (1..L), align y with lagged X, refit SARIMAX as model2, forecast first H=100 on test. #

import numpy as np  # arrays #
import plotly.graph_objects as go  # plots #
from statsmodels.tsa.statespace.sarimax import SARIMAX  # model #

L = 10  # number of lags for exog (try 5,10,20) #
H = 100  # forecast horizon on test #

def add_lags_exog(X, L):  # build lagged exog #
    Xl = pd.concat([X.shift(l).add_suffix(f"_lag{l}") for l in range(0, L+1)], axis=1)  # lag0..lagL #
    return Xl  # dataframe #

# build lagged exog on full cleaned series (keeps alignment) #
X2_lag = add_lags_exog(X2, L)  # (N, 6*(L+1)) #
maskL = X2_lag.notna().all(axis=1) & y2.notna()  # drop rows without full lag history #
y2L = y2[maskL]  # aligned y #
X2L = X2_lag[maskL]  # aligned lagged exog #

N2L = len(y2L)  # samples #
split2L = N2L // 2  # time split #

y_train2L, y_test2L = y2L.iloc[:split2L], y2L.iloc[split2L:]  # train/test y #
X_train2L, X_test2L = X2L.iloc[:split2L], X2L.iloc[split2L:]  # train/test X #

# reuse d2 (0 from ADF), re-search small (p,q) #
best_aic2L = np.inf  # init #
best_order2L = None  # init #

for p2 in range(0, 4):  # p=0..3 #
    for q2 in range(0, 4):  # q=0..3 #
        try:
            m = SARIMAX(
                y_train2L, exog=X_train2L,  # train only #
                order=(p2, d2, q2),  # (p,d,q) #
                enforce_stationarity=False, enforce_invertibility=False
            ).fit(disp=False)  # fit #
            if m.aic < best_aic2L:  # keep best #
                best_aic2L, best_order2L = m.aic, (p2, d2, q2)  # store #
        except Exception:
            pass  # skip #

print("best_order2L:", best_order2L, "best_aic2L:", best_aic2L, "L:", L)  # report #

model2 = SARIMAX(
    y_train2L, exog=X_train2L,  # train only #
    order=best_order2L,  # best #
    enforce_stationarity=False, enforce_invertibility=False
).fit(disp=False)  # final fit #

# forecast first H points of test #
y_test2L_h = y_test2L.iloc[:H]  # true #
X_test2L_h = X_test2L.iloc[:H]  # exog #

pred2_h = model2.get_forecast(steps=H, exog=X_test2L_h)  # forecast #
yhat2_h = pred2_h.predicted_mean  # preds #
history2 = pred2_h  # store #

err2_h = (y_test2L_h.values - yhat2_h.values)  # residuals #
mae2_h = float(np.mean(np.abs(err2_h)))  # MAE #
mse2_h = float(np.mean(err2_h**2))  # MSE #

print("H:", H, "MAE2:", mae2_h, "MSE2:", mse2_h)  # metrics #

fig = go.Figure()  # init #
fig.add_scatter(x=y_test2L_h.index, y=y_test2L_h.values, mode="lines", name="Y_true")  # true #
fig.add_scatter(x=y_test2L_h.index, y=yhat2_h.values,   mode="lines", name="Y_pred")  # pred #
fig.update_layout(template="plotly_dark", title=f"SARIMAX+exog lags (L={L}) | order={best_order2L} | MAE={mae2_h:.4f} MSE={mse2_h:.6f}", xaxis_title="TIME", yaxis_title="Y")  # layout #
fig.show()  # render #


best_order2L: (1, 0, 1) best_aic2L: -34606.77348232396 L: 10
H: 100 MAE2: 0.003421125949961886 MSE2: 1.6847671281834552e-05


In [37]:
# Foreword: single-origin SARIMAX forecast -> pick time sample n in TEST, forecast H=100 ahead (like LSTM horizon), plot. #

import numpy as np  # arrays #
import plotly.graph_objects as go  # plots #

H = 100  # horizon #
n = 1000  # test offset (0 = start of test); set e.g. 200, 500 #

# choose which data to use (with lags if you built them, else falls back to non-lag exog) #
yT = y_test2L if "y_test2L" in globals() else y_test2  # target test series #
XT = X_test2L if "X_test2L" in globals() else X_test2  # exog test matrix #

start_idx = yT.index[n]  # start time #
y_true_h = yT.loc[start_idx:].iloc[:H]  # true Y over horizon #
X_h = XT.loc[start_idx:].iloc[:H]  # exog over horizon (must be known) #

pred_h = model2.get_forecast(steps=len(y_true_h), exog=X_h)  # forecast from origin #
y_pred_h = pred_h.predicted_mean  # preds #

print("start TIME:", start_idx, "H:", len(y_true_h))  # info #

fig = go.Figure()  # init #
fig.add_scatter(x=y_true_h.index, y=y_true_h.values, mode="lines", name="Y_true")  # true #
fig.add_scatter(x=y_true_h.index, y=y_pred_h.values, mode="lines", name="Y_pred")  # pred #
fig.update_layout(template="plotly_dark", title=f"SARIMAX single-origin forecast | n={n} | H={len(y_true_h)}", xaxis_title="TIME", yaxis_title="Y")  # layout #
fig.show()  # render #


start TIME: 4005 H: 100


In [42]:
# Foreword: SARIMAX forecast on a provided step scenario (fresh X), like your LSTM window demo -> build step exog, add lags if model2 needs them, forecast Y horizon H=100, Plotly dark. #

import numpy as np  # arrays #
import pandas as pd  # tables #
import plotly.graph_objects as go  # plots #

H = 100  # horizon length #
t0 = 10  # step time (0..H-1); set 0 for immediate step #

# step levels (before -> after)  # set these like in your screenshot #
X1OUT_0, X1OUT_1 = 0.93, 0.93  # OUT1 #
X2OUT_0, X2OUT_1 = 0.12, 0.12  # OUT2 #
X3OUT_0, X3OUT_1 = 0.20, 0.20  # OUT3 #
X1SP_0,  X1SP_1  = 1.00, 0.80  # SP1 #
X2SP_0,  X2SP_1  = 0.20, 0.40  # SP2 #
X3SP_0,  X3SP_1  = 0.20, 0.40  # SP3 #

# build horizon arrays (H,) #
X1OUT_f = np.r_[np.full(t0, X1OUT_0), np.full(H-t0, X1OUT_1)].astype(float)  # X1OUT #
X1SP_f  = np.r_[np.full(t0, X1SP_0 ), np.full(H-t0, X1SP_1 )].astype(float)  # X1SP #
X2OUT_f = np.r_[np.full(t0, X2OUT_0), np.full(H-t0, X2OUT_1)].astype(float)  # X2OUT #
X2SP_f  = np.r_[np.full(t0, X2SP_0 ), np.full(H-t0, X2SP_1 )].astype(float)  # X2SP #
X3OUT_f = np.r_[np.full(t0, X3OUT_0), np.full(H-t0, X3OUT_1)].astype(float)  # X3OUT #
X3SP_f  = np.r_[np.full(t0, X3SP_0 ), np.full(H-t0, X3SP_1 )].astype(float)  # X3SP #

# fresh exog base (H,6) in model order #
X_fresh0 = pd.DataFrame({
    "X1OUT": X1OUT_f, "X1SP": X1SP_f,
    "X2OUT": X2OUT_f, "X2SP": X2SP_f,
    "X3OUT": X3OUT_f, "X3SP": X3SP_f
}).astype(float)  # (H,6) #

# infer L from model2 training exog (if lagged SARIMAX was used) #
L = 0  # default #
if "X_train2L" in globals():  # lagged exog case #
    L = max([int(c.split("_lag")[-1]) for c in X_train2L.columns if "_lag" in c] + [0])  # infer L #

# build exog for forecasting (needs past context if lagged) #
if L > 0:  # lagged SARIMAX expects columns like *_lag0..*_lagL #
    X_past = df[["X1OUT","X1SP","X2OUT","X2SP","X3OUT","X3SP"]].iloc[-L:].astype(float)  # last L rows as context #
    X_ctx = pd.concat([X_past, X_fresh0], axis=0)  # (L+H,6) #
    X_fresh = pd.concat([X_ctx.shift(l).add_suffix(f"_lag{l}") for l in range(0, L+1)], axis=1).iloc[L:L+H]  # (H,6*(L+1)) #
else:  # non-lag SARIMAX #
    X_fresh = X_fresh0  # (H,6) #

# forecast Y dynamics #
pred = model2.get_forecast(steps=H, exog=X_fresh)  # forecast #
y_pred = pred.predicted_mean  # (H,) #

# plot #
k = np.arange(H)  # step axis #
fig = go.Figure()  # init #
fig.add_scatter(x=k, y=y_pred.values, mode="lines", name="Y_pred")  # predicted Y #
fig.add_vline(x=t0, line_dash="dash")  # step marker #
fig.update_layout(template="plotly_dark", title=f"SARIMAX forecast on step scenario | H={H} | L={L}", xaxis_title="k (step ahead)", yaxis_title="Y")  # layout #
fig.show()  # render #


#RandomForrestRegressor

In [43]:
# Foreword: step 1 (model3) -> build lagged ML dataset: features = exog lags (0..L) + optional Y lags (1..Ly), label = Y(t+1). #

import numpy as np  # arrays #
import pandas as pd  # tables #

L = 10   # exog lags (0..L) #
Ly = 10  # Y lags (1..Ly) #
H = 100  # horizon for later plots #

exog_cols = ["X1OUT","X1SP","X2OUT","X2SP","X3OUT","X3SP"]  # exog #

X0 = df[exog_cols].astype(float)  # exog base #
y0 = df["Y"].astype(float)  # target base #

# exog lags (include lag0) #
X_lags = pd.concat([X0.shift(l).add_suffix(f"_lag{l}") for l in range(0, L+1)], axis=1)  # (N,6*(L+1)) #

# Y lags (autoregressive features; start at 1 to avoid leakage of label) #
y_lags = pd.concat([y0.shift(l).rename(f"Y_lag{l}") for l in range(1, Ly+1)], axis=1)  # (N,Ly) #

# label: next step (t -> t+1) #
y_label = y0.shift(-1).rename("Y_next")  # (N,) #

# final dataset #
data3 = pd.concat([X_lags, y_lags, y_label], axis=1).dropna()  # drop rows with missing lag history #

X3 = data3.drop(columns=["Y_next"])  # features #
y3 = data3["Y_next"]  # label #

N3 = len(data3)  # samples #
split3 = N3 // 2  # time split 50/50 #

X3_train, X3_test = X3.iloc[:split3], X3.iloc[split3:]  # train/test X #
y3_train, y3_test = y3.iloc[:split3], y3.iloc[split3:]  # train/test y #

print("X3:", X3.shape, "y3:", y3.shape)  # shapes #
print("train:", X3_train.shape, y3_train.shape, "test:", X3_test.shape, y3_test.shape)  # split shapes #


X3: (5990, 76) y3: (5990,)
train: (2995, 76) (2995,) test: (2995, 76) (2995,)


In [46]:
# Foreword: step 2 (model3) -> train RandomForestRegressor on lagged dataset (time-split), store history3, evaluate on first H=100 of test, Plotly dark. #

import numpy as np  # arrays #
import pandas as pd  # tables #
import plotly.graph_objects as go  # plots #
from sklearn.ensemble import RandomForestRegressor  # RF regressor #
from sklearn.metrics import mean_absolute_error, mean_squared_error  # metrics #

H = 100  # eval window on test #

model3 = RandomForestRegressor(
    n_estimators=700,  # trees #
    min_samples_leaf=7,  # regularize #
    n_jobs=-1,  # parallel #
    random_state=42  # reproducible #
)  # model3 #

model3.fit(X3_train, y3_train)  # train #

y3_test_h = y3_test.iloc[:H]  # true Y (H) #
X3_test_h = X3_test.iloc[:H]  # test features (H,76) #
yhat3_h = pd.Series(model3.predict(X3_test_h), index=y3_test_h.index)  # preds aligned #

mae3_h = float(mean_absolute_error(y3_test_h, yhat3_h))  # MAE #
mse3_h = float(mean_squared_error(y3_test_h, yhat3_h))  # MSE #
history3 = {"H": H, "MAE": mae3_h, "MSE": mse3_h}  # store #

print("history3:", history3)  # report #

fig = go.Figure()  # init #
fig.add_scatter(x=y3_test_h.index, y=y3_test_h.values, mode="lines", name="Y_true")  # true #
fig.add_scatter(x=y3_test_h.index, y=yhat3_h.values,   mode="lines", name="Y_pred")  # pred #
fig.update_layout(template="plotly_dark", title=f"RF model3 | H={H} | MAE={mae3_h:.4f} MSE={mse3_h:.6f}", xaxis_title="TIME", yaxis_title="Y")  # layout #
fig.show()  # render #


history3: {'H': 100, 'MAE': 0.022476726333580065, 'MSE': 0.0011062929960922196}


In [49]:
# Foreword: model3 fix -> walk-forward (online) ML: refit on last W samples before each step, predict next Y (H=100). This adapts to regime shift at test start. #

import numpy as np  # arrays #
import pandas as pd  # tables #
import plotly.graph_objects as go  # plots #
from sklearn.ensemble import RandomForestRegressor  # RF #
from sklearn.metrics import mean_absolute_error, mean_squared_error  # metrics #

H = 100   # horizon (first H points of test) #
W = 2000  # rolling training window (try 500, 1000, 2000, 3000) #

X3_all = X3  # full features (after dropna) #
y3_all = y3  # full labels (after dropna) #

origin = split3  # test starts here (time split from step 1) #

y_true_wf = y3_all.iloc[origin:origin+H]  # true on horizon #
y_pred_wf = np.empty(H, dtype=float)  # preds #

for i in range(H):  # step-by-step #
    i0 = max(0, origin - W + i)  # window start #
    i1 = origin + i  # window end (exclusive) #
    Xw = X3_all.iloc[i0:i1]  # rolling train X #
    yw = y3_all.iloc[i0:i1]  # rolling train y #
    m3 = RandomForestRegressor(  # small-ish for speed per step #
        n_estimators=300,  # trees #
        min_samples_leaf=2,  # less smoothing than 5 #
        max_features="sqrt",  # diversity #
        n_jobs=-1,  # speed #
        random_state=42  # reproducible #
    )  # model #
    m3.fit(Xw, yw)  # refit #
    y_pred_wf[i] = float(m3.predict(X3_all.iloc[i1:i1+1])[0])  # 1-step pred #

y_pred_wf = pd.Series(y_pred_wf, index=y_true_wf.index)  # align #

mae_wf = float(mean_absolute_error(y_true_wf, y_pred_wf))  # MAE #
mse_wf = float(mean_squared_error(y_true_wf, y_pred_wf))  # MSE #
history3 = {"H": H, "W": W, "MAE": mae_wf, "MSE": mse_wf}  # store #

print("history3_wf:", history3)  # report #

fig = go.Figure()  # init #
fig.add_scatter(x=y_true_wf.index, y=y_true_wf.values, mode="lines", name="Y_true")  # true #
fig.add_scatter(x=y_pred_wf.index, y=y_pred_wf.values, mode="lines", name="Y_pred_wf")  # pred #
fig.update_layout(template="plotly_dark", title=f"Model3 walk-forward RF | H={H} W={W} | MAE={mae_wf:.4f} MSE={mse_wf:.6f}", xaxis_title="TIME", yaxis_title="Y")  # layout #
fig.show()  # render #


history3_wf: {'H': 100, 'W': 2000, 'MAE': 0.007504726991504987, 'MSE': 0.00011289356202705844}


In [54]:
# Foreword: fix RF feature-name mismatch by using EXACT training feature names from model3.feature_names_in_; build row from those names; recursive Y horizon H=100 (Plotly dark). #

import numpy as np  # arrays #
import pandas as pd  # tables #
import plotly.graph_objects as go  # plots #

H = 100  # horizon #
t0 = 10  # step time #

# ====== INPUTS YOU EDIT (before -> after) ====== #
X1OUT_0, X1OUT_1 = 0.93, 0.93  # OUT1 #
X2OUT_0, X2OUT_1 = 0.12, 0.12  # OUT2 #
X3OUT_0, X3OUT_1 = 0.20, 0.20  # OUT3 #
X1SP_0,  X1SP_1  = 1.00, 0.80  # SP1 #
X2SP_0,  X2SP_1  = 0.20, 0.40  # SP2 #
X3SP_0,  X3SP_1  = 0.20, 0.40  # SP3 #
# ============================================== #

def step_sig(v0, v1, H, t0):  # piecewise step #
    return np.r_[np.full(t0, v0), np.full(H - t0, v1)].astype(float)  # (H,) #

# base horizon exog (H,6) in your human names #
Xf = pd.DataFrame({  # (H,6) #
    "X1OUT": step_sig(X1OUT_0, X1OUT_1, H, t0),  # OUT1 #
    "X1SP":  step_sig(X1SP_0,  X1SP_1,  H, t0),  # SP1 #
    "X2OUT": step_sig(X2OUT_0, X2OUT_1, H, t0),  # OUT2 #
    "X2SP":  step_sig(X2SP_0,  X2SP_1,  H, t0),  # SP2 #
    "X3OUT": step_sig(X3OUT_0, X3OUT_1, H, t0),  # OUT3 #
    "X3SP":  step_sig(X3SP_0,  X3SP_1,  H, t0),  # SP3 #
})  # exog #

feat = list(model3.feature_names_in_)  # EXACT names used in fit (e.g., E1_lag0...) #
lag_feats = [c for c in feat if "_lag" in c and not c.startswith("Y_lag")]  # exog-lag features #
y_feats = [c for c in feat if c.startswith("Y_lag")]  # y-lag features #

L = max(int(c.split("_lag")[-1]) for c in lag_feats)  # max exog lag #
Ly = max(int(c.split("Y_lag")[-1]) for c in y_feats) if len(y_feats) else 0  # max y lag #

# map from training exog base names -> our df columns (edit if your training names differ) #
base_map = {  # change keys if model3 used different base names than E1/E2... #
    "E1": "X1OUT",  # e.g., E1 == X1OUT #
    "E2": "X1SP",   # e.g., E2 == X1SP #
    "E3": "X2OUT",  # e.g., E3 == X2OUT #
    "E4": "X2SP",   # e.g., E4 == X2SP #
    "E5": "X3OUT",  # e.g., E5 == X3OUT #
    "E6": "X3SP",   # e.g., E6 == X3SP #
}  # mapping #

# history buffers from real df (robust) #
pre = max(L, Ly) + 1  # history length #
y_buf = df["Y"].astype(float).iloc[-pre:].reset_index(drop=True)  # past Y #
X_hist = df[list(base_map.values())].astype(float).iloc[-pre:].reset_index(drop=True)  # past exog #

X_all = pd.concat([X_hist, Xf.reset_index(drop=True)], axis=0).reset_index(drop=True)  # (pre+H,6) #
y_pred = np.empty(H, dtype=float)  # preds #

for k in range(H):  # recursive #
    t = (pre - 1) + k  # feature time index #
    row = dict.fromkeys(feat, 0.0)  # fill all features (safe default) #

    # fill exog lags using training feature names like "E1_lag7" #
    for c in lag_feats:  # each exog-lag col #
        base, lag = c.split("_lag")  # e.g., ("E1","7") #
        lag = int(lag)  # to int #
        src = base_map.get(base, None)  # map to our exog column #
        if src is not None:  # if mapped #
            row[c] = float(X_all.loc[t - lag, src])  # value #

    # fill Y lags using "Y_lagk" #
    for c in y_feats:  # each y-lag #
        lag = int(c.split("Y_lag")[-1])  # k #
        row[c] = float(y_buf.iloc[-lag])  # y(t-lag) #

    Xrow = pd.DataFrame([row], columns=feat)  # EXACT order #
    y_next = float(model3.predict(Xrow)[0])  # predict #
    y_pred[k] = y_next  # store #
    y_buf = pd.concat([y_buf, pd.Series([y_next])], ignore_index=True)  # append #

# plot #
kk = np.arange(H)  # x #
fig = go.Figure()  # init #
fig.add_scatter(x=kk, y=y_pred, mode="lines", name="Y_pred_RF")  # pred #
fig.add_vline(x=t0, line_dash="dash")  # step marker #
fig.update_layout(template="plotly_dark", title=f"RF (model3) | fresh step inputs -> Y forecast H={H}", xaxis_title="k (step ahead)", yaxis_title="Y")  # layout #
fig.show()  # render #


Porównanie 3 modeli

In [57]:
# Foreword: compare 3 models on the SAME step scenario -> LSTM(model1), SARIMAX(model2), RF(model3); forecast Y for H=100 and plot 3 separate Plotly-dark charts. #

import numpy as np  # arrays #
import pandas as pd  # tables #
import plotly.graph_objects as go  # plots #

H = 100  # horizon #
t0 = 10  # step time (0..H-1) #

# ====== INPUTS YOU EDIT (before -> after) ====== #
X1OUT_0, X1OUT_1 = 0.93, 0.93  # OUT1 #
X2OUT_0, X2OUT_1 = 0.12, 0.12  # OUT2 #
X3OUT_0, X3OUT_1 = 0.20, 0.20  # OUT3 #
X1SP_0,  X1SP_1  = 1.00, 0.80  # SP1 #
X2SP_0,  X2SP_1  = 0.20, 0.60  # SP2 #
X3SP_0,  X3SP_1  = 0.20, 0.90  # SP3 #
# ============================================== #

def step_sig(v0, v1, H, t0):  # piecewise step #
    return np.r_[np.full(t0, v0), np.full(H - t0, v1)].astype(float)  # (H,) #

# base horizon exog in human names (H,6) #
Xf = pd.DataFrame({  # (H,6) #
    "X1OUT": step_sig(X1OUT_0, X1OUT_1, H, t0),  # OUT1 #
    "X1SP":  step_sig(X1SP_0,  X1SP_1,  H, t0),  # SP1 #
    "X2OUT": step_sig(X2OUT_0, X2OUT_1, H, t0),  # OUT2 #
    "X2SP":  step_sig(X2SP_0,  X2SP_1,  H, t0),  # SP2 #
    "X3OUT": step_sig(X3OUT_0, X3OUT_1, H, t0),  # OUT3 #
    "X3SP":  step_sig(X3SP_0,  X3SP_1,  H, t0),  # SP3 #
}).astype(float)  # exog #

# ===================== 1) LSTM (model1) ===================== #
u0 = np.array([X1OUT_0, X1SP_0, X2OUT_0, X2SP_0, X3OUT_0, X3SP_0], dtype=np.float32)  # baseline row #
last_x = np.tile(u0, (T, 1))  # (T,6) constant history #
yhat_lstm = np.empty(H, dtype=np.float32)  # preds #

F = Xf[["X1OUT","X1SP","X2OUT","X2SP","X3OUT","X3SP"]].values.astype(np.float32)  # (H,6) training order #
for k in range(H):  # horizon #
    yhat_lstm[k] = float(model1.predict(last_x.reshape(1, T, D), verbose=0)[0, 0])  # 1-step #
    last_x = np.roll(last_x, -1, axis=0)  # shift #
    last_x[-1, :] = F[k, :]  # append provided features #

# ===================== 2) SARIMAX (model2) ===================== #
L2 = 0  # default #
if "X_train2L" in globals():  # lagged-exog SARIMAX case #
    L2 = max([int(c.split("_lag")[-1]) for c in X_train2L.columns if "_lag" in c] + [0])  # infer L #

if L2 > 0:  # build lagged exog using real past context #
    X_past2 = df[["X1OUT","X1SP","X2OUT","X2SP","X3OUT","X3SP"]].iloc[-L2:].astype(float)  # context #
    X_ctx2 = pd.concat([X_past2, Xf], axis=0).reset_index(drop=True)  # (L2+H,6) #
    X2_exog = pd.concat([X_ctx2.shift(l).add_suffix(f"_lag{l}") for l in range(0, L2+1)], axis=1).iloc[L2:L2+H]  # (H,6*(L2+1)) #
    X2_exog = X2_exog.reindex(columns=X_train2L.columns)  # EXACT training cols/order #
else:  # non-lag exog #
    X2_exog = Xf  # (H,6) #

yhat_sarimax = model2.get_forecast(steps=H, exog=X2_exog).predicted_mean.values.astype(np.float32)  # (H,) #

# ===================== 3) RF (model3) ===================== #
feat3 = list(model3.feature_names_in_)  # EXACT fit-time names #
lag_feats3 = [c for c in feat3 if "_lag" in c and not c.startswith("Y_lag")]  # exog-lag features #
y_feats3 = [c for c in feat3 if c.startswith("Y_lag")]  # y-lag features #

L3 = max(int(c.split("_lag")[-1]) for c in lag_feats3)  # max exog lag #
Ly3 = max(int(c.split("Y_lag")[-1]) for c in y_feats3) if len(y_feats3) else 0  # max y lag #

base_map3 = {  # same as your working RF code #
    "E1": "X1OUT", "E2": "X1SP", "E3": "X2OUT", "E4": "X2SP", "E5": "X3OUT", "E6": "X3SP",
    "X1OUT": "X1OUT", "X1SP": "X1SP", "X2OUT": "X2OUT", "X2SP": "X2SP", "X3OUT": "X3OUT", "X3SP": "X3SP",
}  # mapping #

pre3 = max(L3, Ly3) + 1  # history length #
y_buf3 = df["Y"].astype(float).iloc[-pre3:].reset_index(drop=True)  # past Y #
X_hist3 = df[list({v for v in base_map3.values() if v in Xf.columns})].astype(float).iloc[-pre3:].reset_index(drop=True)  # past exog (human cols) #
X_all3 = pd.concat([X_hist3, Xf.reset_index(drop=True)], axis=0).reset_index(drop=True)  # (pre3+H,6) #

yhat_rf = np.empty(H, dtype=np.float32)  # preds #

for k in range(H):  # recursive #
    t = (pre3 - 1) + k  # time index #
    row = dict.fromkeys(feat3, 0.0)  # safe init #
    for c in lag_feats3:  # exog lags #
        base, lag = c.split("_lag")  # base + lag #
        lag = int(lag)  # int #
        src = base_map3.get(base, None)  # map to human column #
        if src is not None and src in X_all3.columns:  # ok #
            row[c] = float(X_all3.loc[t - lag, src])  # fill #
    for c in y_feats3:  # y lags #
        lag = int(c.split("Y_lag")[-1])  # int #
        row[c] = float(y_buf3.iloc[-lag])  # fill #
    Xrow = pd.DataFrame([row], columns=feat3)  # exact order #
    y_next = float(model3.predict(Xrow)[0])  # predict #
    yhat_rf[k] = y_next  # store #
    y_buf3 = pd.concat([y_buf3, pd.Series([y_next])], ignore_index=True)  # append #

# ===================== PLOTS (3 separate figures) ===================== #
kk = np.arange(H)  # x #

fig1 = go.Figure()  # LSTM #
fig1.add_scatter(x=kk, y=yhat_lstm, mode="lines", name="Y_pred_LSTM")  # pred #
fig1.add_vline(x=t0, line_dash="dash")  # step marker #
fig1.update_layout(template="plotly_dark", title="LSTM (model1) | Y forecast H=100", xaxis_title="k (step ahead)", yaxis_title="Y")  # layout #
fig1.show()  # render #

fig2 = go.Figure()  # SARIMAX #
fig2.add_scatter(x=kk, y=yhat_sarimax, mode="lines", name="Y_pred_SARIMAX")  # pred #
fig2.add_vline(x=t0, line_dash="dash")  # step marker #
fig2.update_layout(template="plotly_dark", title=f"SARIMAX (model2) | Y forecast H=100 | L={L2}", xaxis_title="k (step ahead)", yaxis_title="Y")  # layout #
fig2.show()  # render #

fig3 = go.Figure()  # RF #
fig3.add_scatter(x=kk, y=yhat_rf, mode="lines", name="Y_pred_RF")  # pred #
fig3.add_vline(x=t0, line_dash="dash")  # step marker #
fig3.update_layout(template="plotly_dark", title=f"RandomForest (model3) | Y forecast H=100 | L={L3} Ly={Ly3}", xaxis_title="k (step ahead)", yaxis_title="Y")  # layout #
fig3.show()  # render #


Porównanie predykcji dla 3 modeli z wyborem próbki - aby porównać jaki ma bład

In [59]:
# Foreword: from sample n -> use REAL future inputs X(t) from df to keep dynamics; predict Y for H=100 with 3 models (LSTM=model1, SARIMAX=model2, RF=model3) and plot preds + true Y on ONE Plotly-dark chart. #

import numpy as np  # arrays #
import pandas as pd  # tables #
import plotly.graph_objects as go  # plots #

n = 245  # origin sample index (t = n) #
H = 100  # horizon steps (predict Y at t+1 .. t+H) #

exog_cols = ["X1OUT","X1SP","X2OUT","X2SP","X3OUT","X3SP"]  # training order for exog #

N = len(df)  # length #
H_eff = int(min(H, max(0, (N - 1) - n)))  # ensure we have df rows up to n+H #
assert H_eff > 0, "Za malo danych: n jest za blisko konca df."  # guard #

# --- baseline values at time n (for display) --- #
row_n = df.iloc[n]  # row at t=n #
X1OUT_0 = float(row_n["X1OUT"]); X1SP_0 = float(row_n["X1SP"])  # #
X2OUT_0 = float(row_n["X2OUT"]); X2SP_0 = float(row_n["X2SP"])  # #
X3OUT_0 = float(row_n["X3OUT"]); X3SP_0 = float(row_n["X3SP"])  # #

print("n =", n)  # #
print("X1OUT_0 =", X1OUT_0, "X1SP_0 =", X1SP_0)  # #
print("X2OUT_0 =", X2OUT_0, "X2SP_0 =", X2SP_0)  # #
print("X3OUT_0 =", X3OUT_0, "X3SP_0 =", X3SP_0)  # #
print("H_eff =", H_eff, "| predicts Y at times n+1 .. n+H_eff")  # #

# --- true future Y for comparison (t=n+1..n+H_eff) --- #
idx_h = df.index[n+1:n+1+H_eff]  # TIME index for horizon #
y_true_h = df["Y"].astype(float).iloc[n+1:n+1+H_eff].to_numpy()  # (H_eff,) #

# ===================== 1) LSTM (model1): window from REAL history, then roll with REAL future X ===================== #
# Need last T rows ending at n, in training exog order #
X_exog_all = df[exog_cols].astype(np.float32)  # full exog #
start_hist = max(0, n - (T - 1))  # history start #
X_hist = X_exog_all.iloc[start_hist:n+1].to_numpy()  # (<=T,6) #

if X_hist.shape[0] < T:  # pad if not enough history #
    pad = np.tile(X_hist[0], (T - X_hist.shape[0], 1))  # (pad,6) #
    last_x = np.vstack([pad, X_hist]).astype(np.float32)  # (T,6) #
else:
    last_x = X_hist[-T:].astype(np.float32)  # (T,6) #

X_future = X_exog_all.iloc[n+1:n+1+H_eff].to_numpy().astype(np.float32)  # (H_eff,6) #
yhat_lstm = np.empty(H_eff, dtype=np.float32)  # preds #

for k in range(H_eff):  # #
    yhat_lstm[k] = float(model1.predict(last_x.reshape(1, T, D), verbose=0)[0, 0])  # 1-step pred #
    last_x = np.roll(last_x, -1, axis=0)  # shift window #
    last_x[-1, :] = X_future[k, :]  # append REAL future X #

# ===================== 2) SARIMAX (model2): forecast with REAL future exog (and lags if used in training) ===================== #
X_base_h = df[exog_cols].astype(float).iloc[n+1:n+1+H_eff].copy()  # (H_eff,6) #

L2 = 0  # default no lagged-exog #
if "X_train2L" in globals():  # lagged-exog SARIMAX trained #
    L2 = max([int(c.split("_lag")[-1]) for c in X_train2L.columns if "_lag" in c] + [0])  # infer L #

if L2 > 0:  # build lagged exog aligned to training #
    X_past = df[exog_cols].astype(float).iloc[max(0, n - L2 + 1):n+1].copy()  # up to L2 rows #
    if len(X_past) < L2:  # pad past context if needed #
        pad = pd.concat([X_past.iloc[[0]]] * (L2 - len(X_past)), axis=0)  # repeat first #
        X_past = pd.concat([pad, X_past], axis=0)  # (L2,6) #
    X_ctx = pd.concat([X_past, X_base_h], axis=0)  # (L2+H_eff,6) #
    X_lag = pd.concat([X_ctx.shift(l).add_suffix(f"_lag{l}") for l in range(0, L2+1)], axis=1)  # #
    X_fresh2 = X_lag.iloc[L2:L2+H_eff].copy()  # horizon slice #
    X_fresh2 = X_fresh2.reindex(columns=X_train2L.columns)  # exact training order #
else:
    X_fresh2 = X_base_h  # (H_eff,6) #

yhat_sar = model2.get_forecast(steps=H_eff, exog=X_fresh2).predicted_mean.to_numpy().astype(np.float32)  # #

# ===================== 3) RF (model3): recursive Y with REAL future exog path (needs model3.feature_names_in_) ===================== #
feat = list(model3.feature_names_in_)  # exact fitted feature names #
lag_feats = [c for c in feat if ("_lag" in c and not c.startswith("Y_lag"))]  # exog-lag features #
y_feats = [c for c in feat if c.startswith("Y_lag")]  # y-lag features #

L3 = max([int(c.split("_lag")[-1]) for c in lag_feats] + [0])  # max exog lag #
Ly3 = max([int(c.split("Y_lag")[-1]) for c in y_feats] + [0])  # max y lag #

bases = sorted(set([c.split("_lag")[0] for c in lag_feats]))  # base names used in fit (e.g., X1OUT or E1) #

# build base_map robustly #
if set(bases) == set(exog_cols):  # model trained on X1OUT... #
    base_map = {b: b for b in bases}  # identity #
elif set(["E1","E2","E3","E4","E5","E6"]).issubset(set(bases)):  # model trained on E1..E6 #
    base_map = {"E1":"X1OUT","E2":"X1SP","E3":"X2OUT","E4":"X2SP","E5":"X3OUT","E6":"X3SP"}  # fixed map #
else:  # fallback: map first 6 bases to exog_cols in order #
    base_map = {b: exog_cols[i] for i, b in enumerate(bases[:6])}  # best-effort #

# exog source for RF needs up to time t=n+H_eff-1 and past lags #
X_src = df[exog_cols].astype(float).reset_index(drop=True)  # 0..N-1 #
y_src = df["Y"].astype(float).reset_index(drop=True)  # 0..N-1 #

pre = max(L3, Ly3) + 1  # history needed #
y_buf = y_src.iloc[max(0, n - pre + 1):n+1].copy().reset_index(drop=True)  # true Y history up to n #
if len(y_buf) < pre:  # pad Y history #
    pad = pd.Series([float(y_buf.iloc[0])] * (pre - len(y_buf)))  # #
    y_buf = pd.concat([pad, y_buf], ignore_index=True)  # #

yhat_rf = np.empty(H_eff, dtype=np.float32)  # preds #

for k in range(H_eff):  # predict Y at t=n+k+1 #
    t = n + k  # features at time t predict y(t+1) #
    row = dict.fromkeys(feat, 0.0)  # init all features #

    # fill exog lag features #
    for c in lag_feats:  # e.g., "E1_lag7" or "X1OUT_lag7" #
        base, lag = c.split("_lag")  # #
        lag = int(lag)  # #
        src_col = base_map.get(base, None)  # map to df exog col #
        if src_col is not None:  # #
            tt = max(0, t - lag)  # clamp #
            row[c] = float(X_src.loc[tt, src_col])  # value #

    # fill Y lag features (from buffer, latest at end) #
    for c in y_feats:  # e.g., "Y_lag3" #
        lag = int(c.split("Y_lag")[-1])  # #
        row[c] = float(y_buf.iloc[-lag])  # y(t-lag) #

    Xrow = pd.DataFrame([row], columns=feat)  # exact order #
    y_next = float(model3.predict(Xrow)[0])  # y(t+1) #
    yhat_rf[k] = y_next  # store #
    y_buf = pd.concat([y_buf, pd.Series([y_next])], ignore_index=True)  # extend history #

# ===================== PLOT: true vs 3 preds on one chart ===================== #
fig = go.Figure()  # init #
fig.add_scatter(x=idx_h, y=y_true_h, mode="lines", name="Y_true")  # true #
fig.add_scatter(x=idx_h, y=yhat_lstm, mode="lines", name="LSTM (model1)")  # lstm #
fig.add_scatter(x=idx_h, y=yhat_sar,  mode="lines", name="SARIMAX (model2)")  # sarimax #
fig.add_scatter(x=idx_h, y=yhat_rf,   mode="lines", name="RF (model3)")  # rf #
fig.update_layout(template="plotly_dark", title=f"Y forecast from sample n={n} using REAL future X | H={H_eff}", xaxis_title="TIME", yaxis_title="Y")  # layout #
fig.show()  # render #

# --- scalar compare at +H_eff --- #
y_true_H = float(y_true_h[-1])  # #
print(f"Y_true(+{H_eff}) =", y_true_H)  # #
print(f"Yhat_LSTM(+{H_eff}) =", float(yhat_lstm[-1]))  # #
print(f"Yhat_SARIMAX(+{H_eff}) =", float(yhat_sar[-1]))  # #
print(f"Yhat_RF(+{H_eff}) =", float(yhat_rf[-1]))  # #


n = 245
X1OUT_0 = 0.926924298471125 X1SP_0 = 1.0
X2OUT_0 = 0.127729274186114 X2SP_0 = 0.2
X3OUT_0 = 0.196273003792552 X3SP_0 = 0.2
H_eff = 100 | predicts Y at times n+1 .. n+H_eff


Y_true(+100) = 0.466372083639611
Yhat_LSTM(+100) = 0.4676581621170044
Yhat_SARIMAX(+100) = 0.4585418403148651
Yhat_RF(+100) = 0.4103955328464508
